# Captura Hantek — vista previa (matplotlib)

Lee `captura_scope.csv` generado con:

`python -m hantek_usb.cli get-real-data --parse --export-csv captura_scope.csv --count-a 0x400 --count-b 0`

Las muestras son **ADC 8 bit** (0–255), sin conversión a voltios.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def load_hantek_scope_csv(path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Lee CSV de --export-csv (líneas # y cabecera index,time_s,adc_u8)."""
    rows: list[list[float]] = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("index"):
                continue
            a, b, c = line.split(",")
            rows.append([float(a), float(b), float(c)])
    data = np.asarray(rows, dtype=float)
    return data[:, 0], data[:, 1], data[:, 2].astype(int)


# Busca captura_scope.csv (desde hantek/ o desde hantek/notebooks/)
_here = Path.cwd()
for CSV in (_here / "captura_scope.csv", _here.parent / "captura_scope.csv"):
    if CSV.exists():
        break
else:
    raise FileNotFoundError(
        "No está captura_scope.csv. Generá uno: "
        "python -m hantek_usb.cli get-real-data --parse --export-csv captura_scope.csv "
        "--count-a 0x400 --count-b 0"
    )

_, time_s, adc = load_hantek_scope_csv(CSV)

print(f"Muestras: {len(adc)}  |  ADC min={adc.min()}  max={adc.max()}")
print(f"CSV: {CSV.resolve()}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time_s, adc, lw=0.8, color="C0")
ax.set_xlabel("time_s (por defecto 1 s entre muestras; ajustá --csv-dt en la captura)")
ax.set_ylabel("adc_u8 (crudo)")
ax.set_title("Captura osciloscopio Hantek")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()